[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivtx/ai-anatomy/blob/main/notebooks/48_lr_cooldown.ipynb)

# 48 · The cooldown, measured: schedules, branches and late data
Train a small character-level language model with four learning-rate schedules and watch the loss drop when
the learning rate comes down. Then branch cooldowns off a constant-LR run at three points, and finally test my
conjecture that the cooldown's data shapes the final model about as much as the first half of training.
Chapter: **The cooldown: why loss drops at the end**.

**Hardware:** CPU, about 5 minutes (a GPU makes it seconds).

**You're done when** you have the four schedule curves, the branched cooldowns, and a verdict on the
late-data conjecture with its numbers.

In [ ]:
import os
import math
import inspect
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import argparse
import textwrap
import json.decoder
import json.encoder
import csv
import calendar
import difflib
import heapq
import shlex
import string
import pprint
import fractions
import statistics

QUICK = os.environ.get("QUICK") == "1"
torch.manual_seed(0)
np.random.seed(0)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not os.path.exists("shakespeare.txt"):
    urllib.request.urlretrieve(url, "shakespeare.txt")

A_text = open("shakespeare.txt").read()  # corpus A: Shakespeare
B_text = "\n".join(
    inspect.getsource(m)
    for m in [
        argparse,
        textwrap,
        json.decoder,
        json.encoder,
        csv,
        calendar,
        difflib,
        heapq,
        shlex,
        string,
        pprint,
        fractions,
        statistics,
    ]
)  # corpus B: Python code
chars = sorted(set(A_text) | set(B_text))
stoi = {c: i for i, c in enumerate(chars)}
V = len(chars)


def tensor(t):
    return torch.tensor([stoi[c] for c in t])


A, B = tensor(A_text), tensor(B_text)
A_tr, A_va = A[: int(0.9 * len(A))], A[int(0.9 * len(A)) :]
B_tr, B_va = B[: int(0.9 * len(B))], B[int(0.9 * len(B)) :]
print(f"Shakespeare {len(A):,} chars, Python {len(B):,} chars, vocabulary {V}")
CTX, STEPS, BS = 32, (600 if QUICK else 3000), 64

## A small model and a training loop with a schedule and a data mix
An MLP over the last 32 characters. `mix(step)` gives the share of each batch drawn from corpus A.

In [ ]:
class CharMLP(nn.Module):
    def __init__(self, d=24, h=512):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.net = nn.Sequential(
            nn.Linear(CTX * d, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, V)
        )

    def forward(self, x):
        return self.net(self.emb(x).flatten(1))


def batch(src, n, g):
    if n == 0:
        return torch.zeros(0, CTX, dtype=torch.long), torch.zeros(0, dtype=torch.long)
    ix = torch.randint(0, len(src) - CTX - 1, (n,), generator=g)
    return torch.stack([src[i : i + CTX] for i in ix]), src[ix + CTX]


def get(n, a_share, g):
    na = int(round(n * a_share))
    xa, ya = batch(A_tr, na, g)
    xb, yb = batch(B_tr, n - na, g)
    return torch.cat([xa, xb]), torch.cat([ya, yb])


@torch.no_grad()
def val(m, src, n=2000):
    g = torch.Generator().manual_seed(123)
    x, y = batch(src, n, g)
    return F.cross_entropy(m(x), y).item()


def lr_at(kind, t, T, decay=0.2, start=None):
    w = max(1, T // 50)
    if t < w:
        return (t + 1) / w
    if kind == "constant":
        return 1.0
    if kind == "cosine":
        return 0.5 * (1 + math.cos(math.pi * (t - w) / (T - w)))
    if kind == "linear":
        return 1 - (t - w) / (T - w)
    if kind == "wsd":
        s = int(T * (1 - decay))
        return 1.0 if t < s else max(0.0, (T - t) / (T - s))


def train(
    kind="cosine",
    mix=lambda t: 0.5,
    T=STEPS,
    peak=3e-3,
    seed=0,
    model=None,
    opt=None,
    t0=0,
    every=50,
    keep=(),
):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    m = model or CharMLP()
    opt = opt or torch.optim.AdamW(m.parameters(), lr=peak, weight_decay=0.1)
    curve, saved = [], {}
    for t in range(t0, T):
        for gr in opt.param_groups:
            gr["lr"] = peak * lr_at(kind, t, T)
        x, y = get(BS, mix(t), g)
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (t + 1) % every == 0 or t + 1 == T:
            curve.append((t + 1, 0.5 * (val(m, A_va, 500) + val(m, B_va, 500))))
        if t + 1 in keep:
            saved[t + 1] = (
                {k: v.clone() for k, v in m.state_dict().items()},
                {k: v for k, v in opt.state_dict().items()},
            )
    return m, np.array(curve), saved

## 1. Four schedules, same everything else
**Prediction (~75%):** constant ends clearly worst; cosine, linear-to-zero and WSD (20% cooldown) finish
within 1% of each other in validation loss, and WSD's curve drops sharply in its last 20%.

In [ ]:
results = {}

for kind in ["constant", "cosine", "linear", "wsd"]:
    m, curve, saved = train(
        kind, keep=(STEPS // 4, STEPS // 2, 3 * STEPS // 4) if kind == "constant" else ()
    )
    results[kind] = (curve, saved)
    print(f"{kind:9s} final loss {curve[-1, 1]:.4f}")

for k, (c, _) in results.items():
    plt.plot(c[:, 0], c[:, 1], label=k)

plt.xlabel("step")
plt.ylabel("validation loss (avg of both corpora)")
plt.ylim(None, results["constant"][0][len(results["constant"][0]) // 5, 1])
plt.legend()
plt.title("learning-rate schedules")
plt.show()

## 2. Branch cooldowns off the constant run
Take the constant-LR checkpoint at 25%, 50% and 75% of training, and cool each down linearly over an extra 20%
of its own length. **Prediction (~70%):** each branch lands within 1.5% of a cosine run of the same total
length, so one constant run gives you a finished model for every budget.

In [ ]:
branches = {}

for at, (state, ostate) in results["constant"][1].items():
    T_b = int(at * 1.2)
    m = CharMLP()
    m.load_state_dict(state)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=0.1)
    opt.load_state_dict(ostate)
    _, cb, _ = train("wsd", T=T_b, model=m, opt=opt, t0=at)
    ref = train("cosine", T=T_b)[1][-1, 1]
    branches[at] = (cb, ref)
    print(
        f"branch at {at}: cooled to {cb[-1, 1]:.4f}; cosine run of {T_b} steps: {ref:.4f} ({100 * (cb[-1, 1] / ref - 1):+.1f}%)"
    )

c = results["constant"][0]
plt.plot(c[:, 0], c[:, 1], c="gray", label="constant")

for at, (cb, ref) in branches.items():
    plt.plot(cb[:, 0], cb[:, 1], label=f"cooldown from step {at}")
    plt.scatter([cb[-1, 0]], [ref], marker="x", c="k")

plt.xlabel("step")
plt.ylabel("validation loss")
plt.legend()
plt.title("branched cooldowns (x = cosine run of the same length)")
plt.show()

## 3. The late-data conjecture
Every run sees exactly 50% Shakespeare (A) and 50% Python (B) overall, with a WSD schedule. They differ in
*when* the extra Shakespeare comes:
* **even:** 50% A throughout.
* **early-A:** 70% A in the first half, 30% in the second (10% of all tokens moved early).
* **cool-A:** 100% A in the 20% cooldown, 37.5% before (also 10% of all tokens moved, but late).

The measure is how much better the model is at A relative to B, compared with *even*:
`pref = (loss_B − loss_A) − (loss_B − loss_A)_even`. **My conjecture (~65%):** cool-A's preference for A is at
least as large as early-A's, even though early-A's extra A arrived when the learning rate was high and the model
was learning fastest.

In [ ]:
mixes = {
    "even": lambda t: 0.5,
    "early-A": lambda t: 0.7 if t < STEPS // 2 else 0.3,
    "cool-A": lambda t: 1.0 if t >= int(STEPS * 0.8) else 0.375,
}
gap = {}

for name, mix in mixes.items():
    vals = []
    for seed in ([0] if QUICK else [0, 1, 2]):
        m, _, _ = train("wsd", mix=mix, seed=seed)
        vals.append((val(m, A_va), val(m, B_va)))
    la, lb = np.mean(vals, 0)
    gap[name] = lb - la
    print(f"{name:8s} loss A {la:.4f}  loss B {lb:.4f}")

pref = {k: gap[k] - gap["even"] for k in mixes}
print("preference for Shakespeare relative to even:", {k: round(v, 4) for k, v in pref.items()})
print(
    "verdict:",
    (
        "conjecture holds"
        if pref["cool-A"] >= pref["early-A"]
        else "conjecture fails: early data mattered more"
    ),
)

**What I measured** (full run, CPU, 3,000 steps; 3 seeds for part 3):

| | final validation loss |
|---|---|
| constant | 2.309 |
| cosine | **2.184** |
| linear to zero | 2.189 |
| WSD, 20% cooldown | 2.214 |

* **Mostly right:** constant was clearly worst, and cosine and linear finished within 0.3%. WSD came in 1.4%
  behind cosine, just outside my 1% prediction. On a run this short, a 600-step cooldown is short in absolute
  terms; WSD's published parity is at much longer runs.
* **Right:** branched cooldowns matched cosine runs of the same length within 1% (−0.3%, 0.0%, −0.8%).
  One constant run really does give you a finished model at every budget.
* **Right, and stronger than I expected:** the late-data test. Moving 10% of tokens into the cooldown shifted the
  model toward Shakespeare by +0.30 nats (A vs B gap, relative to *even*); moving the same amount into the first
  half shifted it the *other* way (−0.10), because that run's second half was Python-heavy and the end of
  training won again. At this scale, the model resembles what it saw last.

This is a small character model; whether the effect is as large at billions of parameters, where models retain
more of what they learned early, is exactly the open question.

**Stretch:** repeat part 3 with a cosine schedule. If the conjecture is about the learning rate, cosine's slow
decay should spread the late-data effect out and weaken it.